# LDA Topic Modelling

## What it does
**Latent Dirichlet Allocation (LDA)** is a generative probabilistic model that discovers
latent topics in a text corpus. Each document is modelled as a mixture of topics,
and each topic as a distribution over words:

$$p(w | d) = \sum_{k=1}^{K} p(w | z=k) \cdot p(z=k | d)$$

where $z$ are latent topics, $K$ is the number of topics chosen by the user.

## Pipeline
1. **Load & aggregate** — tokenize raw articles, concatenate text per period
2. **CountVectorizer** — build term-document matrix
3. **LDA** — fit topic model; each document gets a topic proportion vector
4. **Inspect topics** — top words per topic for interpretation
5. **Export** — save `topics.csv` with one row per period and K topic proportion columns
   (compatible with `text_lasso.ipynb` which consumes pre-computed features)

## Output format (`topics.csv`)
| date | topic_0 | topic_1 | … | topic_K-1 |
|---|---|---|---|---|
| 2000-01-01 | 0.12 | 0.31 | … | 0.08 |

This CSV can be fed directly into `text_lasso.ipynb` as the feature file.

## Data format
CSV with at least a date column and a text column (one row per article or one row per period).

## Configuration

In [ ]:
CONFIG = {
    # --- Text corpus ---
    'TEXT_FILE':        '../../data/articles.csv',
    'TEXT_DATE_COL':    'date',
    'TEXT_COL':         'text',
    # --- CountVectorizer ---
    'MAX_FEATURES':     5000,     # vocabulary size
    'MIN_DF':           5,        # ignore terms in fewer than N documents
    'MAX_DF':           0.95,     # ignore terms in more than 95% of documents
    'NGRAM_RANGE':      [1, 1],   # unigrams only (LDA works best with unigrams)
    # --- LDA ---
    'N_TOPICS':         10,       # number of latent topics K
    'N_TOP_WORDS':      15,       # words to display per topic for inspection
    'LDA_MAX_ITER':     20,
    'LDA_RANDOM_STATE': 42,
    'LDA_LEARNING_METHOD': 'online',   # 'online' (fast) or 'batch'
    # --- Output ---
    'SAVE_RESULTS':     True,
    'OUTPUT_DIR':       'results',
    'TOPICS_FILE':      'topics.csv',  # consumed by text_lasso.ipynb
}

print('Configuration loaded.')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## Step 1 — Load Text Corpus & Aggregate to Monthly

In [ ]:
import sys, warnings, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('../../').resolve()))
from utils import load_csv

text_df = load_csv(CONFIG['TEXT_FILE'], parse_dates=[CONFIG['TEXT_DATE_COL']])
text_df[CONFIG['TEXT_DATE_COL']] = (
    text_df[CONFIG['TEXT_DATE_COL']].dt.to_period('M').dt.to_timestamp()
)

print(f'Text corpus : {text_df.shape[0]:,} rows')
print(f'Period      : {text_df[CONFIG["TEXT_DATE_COL"]].min().date()} — '
      f'{text_df[CONFIG["TEXT_DATE_COL"]].max().date()}')
print(f'Sample text : {str(text_df[CONFIG["TEXT_COL"]].iloc[0])[:120]}...')

In [ ]:
# Aggregate: concatenate all article texts within each month
monthly_text = (
    text_df.groupby(CONFIG['TEXT_DATE_COL'])[CONFIG['TEXT_COL']]
    .apply(lambda texts: ' '.join(texts.dropna().astype(str)))
    .reset_index()
    .rename(columns={CONFIG['TEXT_DATE_COL']: 'date'})
)

print(f'Monthly periods : {len(monthly_text)}')
print(f'Avg words/month : {monthly_text[CONFIG["TEXT_COL"]].str.split().str.len().mean():.0f}')
monthly_text.head()

## Step 2 — Build Term-Document Matrix

In [ ]:
vectorizer = CountVectorizer(
    max_features = CONFIG['MAX_FEATURES'],
    ngram_range  = tuple(CONFIG['NGRAM_RANGE']),
    min_df       = CONFIG['MIN_DF'],
    max_df       = CONFIG['MAX_DF'],
    stop_words   = 'english',
)

count_matrix = vectorizer.fit_transform(monthly_text[CONFIG['TEXT_COL']])
vocab        = vectorizer.get_feature_names_out()

print(f'Vocabulary size      : {len(vocab):,}')
print(f'Term-document matrix : {count_matrix.shape}  (periods × terms)')
print(f'Matrix sparsity      : {1 - count_matrix.nnz / np.prod(count_matrix.shape):.2%}')

## Step 3 — Fit LDA Topic Model

In [ ]:
lda = LatentDirichletAllocation(
    n_components     = CONFIG['N_TOPICS'],
    max_iter         = CONFIG['LDA_MAX_ITER'],
    learning_method  = CONFIG['LDA_LEARNING_METHOD'],
    random_state     = CONFIG['LDA_RANDOM_STATE'],
    n_jobs           = -1,
)

print(f'Fitting LDA with K={CONFIG["N_TOPICS"]} topics...')
topic_matrix = lda.fit_transform(count_matrix)   # shape: (n_periods, K)

print(f'Done. Perplexity (lower = better fit) : {lda.perplexity(count_matrix):.1f}')
print(f'Log-likelihood                         : {lda.score(count_matrix):.1f}')
print(f'Topic matrix shape                     : {topic_matrix.shape}  (periods × topics)')

## Step 4 — Inspect Topics (Top Words per Topic)

In [ ]:
n_top = CONFIG['N_TOP_WORDS']
topic_labels = []

print(f'Top {n_top} words per topic:')
print('=' * 60)
for topic_idx, topic_vec in enumerate(lda.components_):
    top_word_indices = topic_vec.argsort()[::-1][:n_top]
    top_words = [vocab[i] for i in top_word_indices]
    label = f'topic_{topic_idx}'
    topic_labels.append(label)
    print(f'  {label}: {" | ".join(top_words)}')

In [ ]:
# Heatmap: topic proportions over time
topics_df = pd.DataFrame(
    topic_matrix,
    index   = monthly_text['date'],
    columns = topic_labels,
)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(topics_df.T.values, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax.set_yticks(range(CONFIG['N_TOPICS']))
ax.set_yticklabels(topic_labels, fontsize=9)
n_dates = len(monthly_text)
tick_step = max(1, n_dates // 10)
ax.set_xticks(range(0, n_dates, tick_step))
ax.set_xticklabels(
    [str(monthly_text['date'].iloc[i].date()) for i in range(0, n_dates, tick_step)],
    rotation=45, ha='right', fontsize=8,
)
ax.set(xlabel='Period', title=f'Topic Proportions Over Time (K={CONFIG["N_TOPICS"]})')
plt.colorbar(im, ax=ax, label='Topic proportion')
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart of top words for each topic
n_cols = min(5, CONFIG['N_TOPICS'])
n_rows = int(np.ceil(CONFIG['N_TOPICS'] / n_cols))
n_show = min(10, n_top)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.5, n_rows * 3))
axes = np.array(axes).flatten()

for topic_idx, topic_vec in enumerate(lda.components_):
    top_idx   = topic_vec.argsort()[::-1][:n_show]
    top_words = [vocab[i] for i in top_idx]
    top_scores = topic_vec[top_idx] / topic_vec[top_idx].sum()

    ax = axes[topic_idx]
    ax.barh(range(n_show), top_scores[::-1], color='steelblue', alpha=0.8)
    ax.set_yticks(range(n_show))
    ax.set_yticklabels(top_words[::-1], fontsize=8)
    ax.set_title(f'topic_{topic_idx}', fontsize=10)
    ax.grid(True, alpha=0.3, axis='x')

# Hide unused axes
for i in range(CONFIG['N_TOPICS'], len(axes)):
    axes[i].set_visible(False)

plt.suptitle(f'Top {n_show} Words per Topic', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## Step 5 — Topic Proportion Time Series

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for col in topics_df.columns:
    ax.plot(topics_df.index, topics_df[col], linewidth=1, alpha=0.8, label=col)
ax.set(xlabel='Date', ylabel='Topic proportion',
       title=f'Monthly Topic Proportions (K={CONFIG["N_TOPICS"]})')
ax.legend(ncol=min(5, CONFIG['N_TOPICS']), fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nTopic proportion summary:')
print(topics_df.describe().round(4).to_string())

## Step 6 — Save Topic Proportions & Model Metadata

The output `topics.csv` is compatible with `text_lasso.ipynb`:
set `FEATURE_FILE = 'results/topics.csv'` and `DATE_COL = 'date'` in that notebook.

In [ ]:
if CONFIG['SAVE_RESULTS']:
    os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)

    # 1. Topic proportions (main output for text_lasso.ipynb)
    topics_out = topics_df.reset_index().rename(columns={'date': 'date'})
    topics_path = os.path.join(CONFIG['OUTPUT_DIR'], CONFIG['TOPICS_FILE'])
    topics_out.to_csv(topics_path, index=False)
    print(f'Saved topic proportions : {topics_path}')

    # 2. Top words per topic
    top_words_rows = []
    for topic_idx, topic_vec in enumerate(lda.components_):
        top_idx = topic_vec.argsort()[::-1][:n_top]
        top_words_rows.append({
            'topic': f'topic_{topic_idx}',
            'top_words': ' | '.join([vocab[i] for i in top_idx]),
        })
    words_path = os.path.join(CONFIG['OUTPUT_DIR'], 'lda_topic_words.csv')
    pd.DataFrame(top_words_rows).to_csv(words_path, index=False)
    print(f'Saved topic top words   : {words_path}')

    # 3. Summary metadata
    meta = {
        'n_topics':         CONFIG['N_TOPICS'],
        'n_periods':        len(monthly_text),
        'vocab_size':       len(vocab),
        'max_features':     CONFIG['MAX_FEATURES'],
        'lda_perplexity':   lda.perplexity(count_matrix),
        'lda_log_likelihood': lda.score(count_matrix),
        'date_start':       str(monthly_text['date'].min().date()),
        'date_end':         str(monthly_text['date'].max().date()),
        'notebook':         'lda_topics',
    }
    meta_path = os.path.join(CONFIG['OUTPUT_DIR'], 'lda_summary.csv')
    pd.DataFrame([meta]).to_csv(meta_path, index=False)
    print(f'Saved LDA summary       : {meta_path}')
    print()
    print('To use in text_lasso.ipynb, set:')
    print(f'  FEATURE_FILE = "{topics_path}"')
    print(f'  DATE_COL     = "date"')
else:
    print('SAVE_RESULTS = False — skipping.')